## Temporal Fusion Transformer - Hyperparameter tuning (lookback=24)

From our earlier experiments with different models, we came to the conclusion that it is better to focus on the time period for which the forecast data is available ( 2024 onwards).  Hence we will only consider data for that period and train a deep learning model based on temporal fusion transformer. For this purpose, we will be using the PyTorch Forecasting lobrary.

In [1]:
!pip install -q pytorch-forecasting pytorch-lightning lightning torchmetrics
!pip install -q scikit-learn pandas numpy matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.3/425.3 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 18.2 MB/s eta 0:00:00


In [6]:
# General
import gc
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
import json
from pathlib import Path
import pickle
import itertools

import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor
from lightning.pytorch.loggers import TensorBoardLogger

# From PyTorch
import torch
from torch.nn import BCEWithLogitsLoss
from torchmetrics.classification import AveragePrecision
from lightning.pytorch.callbacks import ModelCheckpoint

# From PyTorch Forecasting
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import CrossEntropy, Metric
from pytorch_forecasting.models.temporal_fusion_transformer.tuning import optimize_hyperparameters
from pytorch_forecasting.tuning import Tuner
from pytorch_forecasting.metrics import MultiHorizonMetric

# From sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, average_precision_score,precision_recall_curve,confusion_matrix,
classification_report)

from google.colab import drive
drive.mount("/content/drive")

# Selecting device
def get_device():
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"

device = get_device()
print(f"Using device ; {device}")

if device=='cuda':
    print(torch.cuda.get_device_name(0))

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    pl.seed_everything(seed, workers=True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

def clean_up():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

class BCEWithLogitsLossWrapper(MultiHorizonMetric):
    def __init__(self,pos_weight=None):
        super().__init__()
        self.criterion = BCEWithLogitsLoss(pos_weight=pos_weight)

    def loss(self, y_pred, target):
        # if model outputs [batch, 1] or [batch, 1, 1]
        y_pred = y_pred.squeeze(-1)
        return self.criterion(y_pred, target.float())


class PRAUCMetric(Metric):
    def __init__(self):
        super().__init__()
        self.metric = AveragePrecision(task="binary")

    def update(self, preds, target):


        preds = torch.sigmoid(preds.squeeze(-1)).view(-1)


        if isinstance(target, (tuple, list)):
            target = target[0]

        target = target.view(-1)
        target = target.long()

        self.metric.update(preds, target)

    def compute(self):
        return self.metric.compute()


ROOT = Path("/content/drive/MyDrive/TFT_Hyperparameter_Search_24")
ROOT.mkdir(exist_ok=True)
CHECKPOINT_DIR = ROOT/"checkpoints"
CHECKPOINT_DIR.mkdir(exist_ok=True)
LOG_DIR = ROOT/"logs"
LOG_DIR.mkdir(exist_ok=True)

# path where the input files are stored
BASE = "/content/drive/MyDrive/Deep_learning_2026"


lookback = 24
learning_rate = 3e-4
max_epochs = 30
patience = 5
batch_size = 64
max_prediction_length = 1

# Relevant timestamps
valid_start = pd.Timestamp("2025-07-01", tz="UTC")
test_start = pd.Timestamp("2025-12-01", tz="UTC")


INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device ; cuda
Tesla T4


In [7]:
import os

print(os.listdir("/content/drive/MyDrive/Deep_learning_2026"))

['forecast_data_2024-03-01_2026-05-02.csv', 'feature_combined_data_2019-05-01_2026-05-01.csv', 'TFT_hyper_24.ipynb']


### Creating dataset and dataloaders

In [8]:
# Loading processed data
df_feature_file = (pd.read_csv(f"{BASE}/feature_combined_data_2019-05-01_2026-05-01.csv",parse_dates=["timestamp_utc"]
                            ).set_index("timestamp_utc").sort_index())
df_forecast = ( pd.read_csv(f"{BASE}/forecast_data_2024-03-01_2026-05-02.csv",parse_dates=["timestamp_utc"]
                            ).set_index("timestamp_utc").sort_index())

df_feature_file['time_idx'] =  np.arange(len(df_feature_file))
calendar_cols = ["hour_sin","hour_cos","day_of_week_sin","day_of_week_cos","month_sin","month_cos",]

observed_cols = [c for c in df_feature_file.columns if c.startswith("observed_")]
forecast_cols = [c for c in df_forecast.columns if c.startswith("forecast_")]
smard_input_cols = [c for c in df_feature_file.columns if c not in calendar_cols + ["Negative_price"] + observed_cols]

observed_weather = df_feature_file[observed_cols].rename(columns=lambda c: c.replace("observed_", ""))
forecast_weather = df_forecast[forecast_cols].rename(columns=lambda c: c.replace("forecast_", ""))

if sorted(observed_weather.columns) != sorted(forecast_weather.columns):
    raise ValueError("Observed and forecast weather columns do not match after removing prefixes")

target = "Negative_price"
weather_cols = list(df_forecast.columns)
reals = calendar_cols + smard_input_cols + weather_cols
scalers = {feature: StandardScaler()for feature in reals}
time_varying_known_reals = (calendar_cols + weather_cols)
time_varying_unknown_reals = smard_input_cols



In [9]:
metadata = {
    "calendar_cols": calendar_cols,
    "smard_input_cols": smard_input_cols,
    "weather_cols": weather_cols,
    "time_varying_known_reals": time_varying_known_reals,
    "time_varying_unknown_reals": time_varying_unknown_reals,
    "target": target,
}

metadata_path = ROOT/"metadata.pkl"

with open(metadata_path,"wb") as f:
    pickle.dump(metadata,f)

print(f"Metdata saved in :{metadata_path}")

Metdata saved in :/content/drive/MyDrive/TFT_Hyperparameter_Search_24/metadata.pkl


In [10]:
def build_tft_dataset(df_feature,df_forecast,calendar_cols,smard_input_cols,test_start,lookback=24):

    # Align time range
    start_date = max(df_feature.index.min(), df_forecast.index.min())

    df = df_feature.loc[start_date:test_start].copy()
    df_weather = df_forecast.loc[start_date:test_start].copy()

    # Remove observed weather
    df = df.drop(columns=[c for c in df.columns if c.startswith("observed_")])

    # Join forecast weather (aligned to time t)
    df = df.join(df_weather, how="inner")

    # Target: predict t+24 negative price
    df["target"] = df["Negative_price"].shift(-24)

    # Shift forecast weather to the same prediction horizon
    forecast_cols = df_forecast.columns
    df[forecast_cols] = df[forecast_cols].shift(-24)

    # Remove rows where either the target or shifted weather is missing
    df = df.dropna(subset=["target"] + list(forecast_cols)).copy()

    # Clean duplicates
    df = df.loc[:, ~df.columns.duplicated()]

    # Rebuild time index
    df = df.drop(columns=["time_idx"], errors="ignore")
    df["time_idx"] = np.arange(len(df))

    df["series"] = 0

    # Feature columns
    cols = (
        calendar_cols
        + smard_input_cols
        + list(df_forecast.columns)
        + ["time_idx", "series", "target"]
    )
    cols = list(dict.fromkeys(cols))

    df_tft = df[cols]

    # Train / full pre-test split
    data = df_tft[df_tft.index < test_start]

    # include history for encoder + horizon
    test_data = df_tft[
        df_tft.index >= (test_start - pd.Timedelta(hours=lookback+24 ))
    ].copy()

    return df_tft, data, test_data

In [11]:
df_tft, data, test_data = build_tft_dataset(df_feature_file,df_forecast,calendar_cols,
                                            smard_input_cols,test_start,lookback=lookback)

train_data = data[data.index < valid_start]

training = TimeSeriesDataSet(train_data,time_idx="time_idx",target="target",group_ids=["series"],
                              max_encoder_length=lookback,max_prediction_length=1,
                              time_varying_known_reals=time_varying_known_reals,
                              time_varying_unknown_reals=time_varying_unknown_reals,
                              add_relative_time_idx=True,add_target_scales=False,add_encoder_length=True,)

validation_data = data[(data.index >= valid_start) & (data.index < test_start)]

validation = TimeSeriesDataSet.from_dataset(training,validation_data,stop_randomization=True)

test_dataset = TimeSeriesDataSet.from_dataset(training,test_data,stop_randomization=True)

# imbalance handling
n_pos = train_data["target"].sum()
n_neg = len(train_data) - n_pos
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32, device=device)

train_loader = training.to_dataloader(train=True, batch_size=batch_size, num_workers=2,persistent_workers=True)
val_loader = validation.to_dataloader(train=False, batch_size=batch_size, num_workers=2,persistent_workers=True)

## Hyperparameter

Early stopping based on PR-AUC. Also using BCEWithLogitsLoss with pos_weight

In [12]:
# Defining parameter grid for hyperparameter tuning

param_grid = list(itertools.product(
    [16, 32],          # hidden_size
    [1, 4],            # attention_head_size
    [0.1, 0.2],        # dropout
    [8, 16],           # hidden_continuous_size
    [3e-4],      # learning_rate
))

columns = ["hidden_size","attention_head_size","dropout","hidden_continuous_size","learning_rate",]

# Defining functions for use in hyperparameter tuning

def train_tft(training_dataset,train_loader,val_loader,hidden_size=16,attention_head_size=4,dropout=0.1,
              hidden_continuous_size=8,learning_rate=3e-4,max_epochs=30,patience=5,pos_weight=None,):

    device = get_device()
    print(f"\nUsing device: {device}")

    tft = TemporalFusionTransformer.from_dataset(training_dataset,learning_rate=learning_rate,hidden_size=hidden_size,
                                                 attention_head_size=attention_head_size,dropout=dropout,
                                                 hidden_continuous_size=hidden_continuous_size,
                                                 loss=BCEWithLogitsLossWrapper(pos_weight=pos_weight),
                                                 logging_metrics=[PRAUCMetric()],output_size=1,)

    early_stop = EarlyStopping(monitor="val_PRAUCMetric",mode="max",patience=patience,)

    lr_logger = LearningRateMonitor()

    checkpoint_callback = ModelCheckpoint(
        dirpath=CHECKPOINT_DIR,
        filename=f"tft_h{hidden_size}_a{attention_head_size}_d{dropout}_c{hidden_continuous_size}",
        monitor="val_PRAUCMetric",
        mode="max",
        save_top_k=1,
    )

    trainer = pl.Trainer(max_epochs=max_epochs,accelerator="auto",devices=1,gradient_clip_val=0.1,
                         callbacks=[early_stop, lr_logger, checkpoint_callback],enable_progress_bar=True,
                         log_every_n_steps=10,)

    trainer.fit(tft,train_dataloaders=train_loader,val_dataloaders=val_loader,)

    return tft

def evaluate_tft(model, val_loader):

    raw = model.predict(val_loader, mode="prediction")

    # Checking tensor format
    probs = torch.sigmoid(raw).detach().cpu().view(-1).numpy()

    y_true = torch.cat([y[0] for _, y in val_loader]).cpu().numpy().reshape(-1)


    roc_auc = roc_auc_score(y_true, probs)
    pr_auc = average_precision_score(y_true, probs)

    precision, recall, thresholds = precision_recall_curve(y_true, probs)

    f1_scores = (2 * precision[:-1] * recall[:-1]/ (precision[:-1] + recall[:-1] + 1e-8))

    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx]
    best_f1 = f1_scores[best_idx]

    preds_binary = (probs >= best_threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, preds_binary).ravel()

    return {"roc_auc": roc_auc,"pr_auc": pr_auc,"best_threshold": best_threshold,"best_f1": best_f1,
            "precision": tp / (tp + fp + 1e-8),"recall": tp / (tp + fn + 1e-8),"confusion_matrix": (tn, fp, fn, tp),}

In [13]:
results = []

results_path = ROOT / "tuning_results_24.csv"


# load existing results if any
if results_path.exists():
    results_df_existing = pd.read_csv(results_path)
    results = results_df_existing.to_dict("records")

    print(f"Resuming from {len(results)} completed trials")
else:
    print("Starting fresh tuning run")


# Helper: check completed configs
def is_done(config):
    for r in results:
        if (
            r["hidden_size"] == config[0]
            and r["attention_head_size"] == config[1]
            and r["dropout"] == config[2]
            and r["hidden_continuous_size"] == config[3]
            and r["learning_rate"] == config[4]
        ):
            return True
    return False

# Grid search loop
for i, params in enumerate(param_grid):

    if is_done(params):
        print(f"[SKIP] Already done: {params}")
        continue

    hidden_size, attn_heads, dropout, hidden_cont_size, lr = params

    print(f"\n[{i+1}/{len(param_grid)}] Running: {params}")

    try:

        tft = train_tft(
            training_dataset=training,
            train_loader=train_loader,
            val_loader=val_loader,
            hidden_size=hidden_size,
            attention_head_size=attn_heads,
            dropout=dropout,
            hidden_continuous_size=hidden_cont_size,
            learning_rate=lr,
            pos_weight=pos_weight,
            max_epochs=max_epochs,
            patience=patience,
        )

        metrics = evaluate_tft(tft, val_loader)

        result = {
            "hidden_size": hidden_size,
            "attention_head_size": attn_heads,
            "dropout": dropout,
            "hidden_continuous_size": hidden_cont_size,
            "learning_rate": lr,
            **metrics,
        }

        results.append(result)


        # save after each trial
        pd.DataFrame(results).to_csv(results_path, index=False)

        print(f"Saved → {results_path}")


        # cleanup
        del tft
        clean_up()

    except Exception as e:
        print(f"FAILED: {params}")
        print(e)

        clean_up()
        continue

Starting fresh tuning run

[1/16] Running: (16, 1, 0.1, 8, 0.0003)

Using device: cuda


/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightn

┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ BCEWithLogitsLossWrapper        │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │      0 │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │    624 │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │    528 │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │ 27.0 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │ 19.3 K │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │  2.2 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │  2.2 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │    544 │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │     32 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │  1.4 K │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │  1.1 K │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │    576 │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │    576 │ train │     0 │
│ 20 │ output_layer                       │ Linear                          │     17 │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 60.5 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 60.5 K                                                                                               
Total estimated model params size (MB): 0.242                                                                      
Modules in train mode: 1103                                                                                        
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

...

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

INFO: `Trainer.fit` stopped: `max_epochs=30` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=30` reached.


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch

Saved → /content/drive/MyDrive/TFT_Hyperparameter_Search_24/tuning_results_24.csv


In [14]:
import os
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
import pandas as pd

results_df = pd.read_csv("/content/drive/MyDrive/TFT_Hyperparameter_Search_24/tuning_results_24.csv")

Mounted at /content/drive


In [15]:
results_df_sorted = results_df.sort_values("pr_auc", ascending=False)
best_row = results_df_sorted.iloc[0]
best_row


,7
hidden_size,16
attention_head_size,4
dropout,0.2
hidden_continuous_size,16
learning_rate,0.0003
roc_auc,0.902797
pr_auc,0.76573
best_threshold,0.924425
best_f1,0.73743
precision,0.758621


In [16]:
best_config = {
    "hidden_size": int(best_row["hidden_size"]),
    "attention_head_size": int(best_row["attention_head_size"]),
    "dropout": float(best_row["dropout"]),
    "hidden_continuous_size": int(best_row["hidden_continuous_size"]),
    "learning_rate": float(best_row["learning_rate"]),
}

best_config

{'hidden_size': 16,
 'attention_head_size': 4,
 'dropout': 0.2,
 'hidden_continuous_size': 16,
 'learning_rate': 0.0003}

In [17]:
from pathlib import Path

CHECKPOINT_DIR = Path("/content/drive/MyDrive/TFT_Hyperparameter_Search_24/checkpoints")

pattern = f"tft_h{best_config['hidden_size']}_a{best_config['attention_head_size']}_d{best_config['dropout']}_c{best_config['hidden_continuous_size']}"

ckpt_path = list(CHECKPOINT_DIR.glob(pattern + "*.ckpt"))[0]

ckpt_path

PosixPath('/content/drive/MyDrive/TFT_Hyperparameter_Search_24/checkpoints/tft_h16_a4_d0.2_c16.ckpt')